# Solar statistical baselines (Darts) — one-year setup


In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"solar"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from darts import TimeSeries, concatenate
from darts.models import NaiveSeasonal, AutoARIMA, AutoTheta, AutoETS
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from preprocessing import load_data
from src.metrics import print_metrics
from power_conversion import convert_solar_result_to_power

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
NUM_SAMPLES=100

TEST_CUTOFF=pd.Timestamp("2023-09-30 23:00")
COVARIATES=["AIRTEMP","RH_AVG","DEWPT","WS","GHI","DNI"]
FUTURE_PROXY_COLS=[f"{c}_lag24" for c in COVARIATES]

LOAD_EXISTING=False
SAVE_MODELS=True
MODEL_DIR=ROOT/"models"/"darts"/"statistical"
OUTPUT_DIR=ROOT/"outputs"/"predictions"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

STAT_CLASSES={"auto_arima":AutoARIMA,"auto_theta":AutoTheta,"auto_ets":AutoETS}

def build_stat(name):
    if name=="auto_arima":
        return AutoARIMA(
            seasonal=True,
            season_length=24,
            stepwise=True,
            trace=True,
            d=1,D=1,
            max_p=1,max_q=1,max_P=1,max_Q=1,
        )
    if name=="auto_theta":
        return AutoTheta(season_length=24)
    if name=="auto_ets":
        return AutoETS(season_length=24)
    raise ValueError(name)

def clip_poa(result):
    forecast_cols=[
        c for c in result.columns
        if c not in {"item_id","timestamp","target"}
    ]
    result=result.copy()
    result[forecast_cols]=result[forecast_cols].clip(lower=0.0)
    if "0.5" in result.columns:
        result["prediction"]=result["0.5"]
    return result

def probabilistic_frame(series,forecast):
    gt=series.slice_intersect(forecast)
    fc=forecast.slice_intersect(gt)
    out=pd.DataFrame({
        "item_id":1,
        "timestamp":gt.time_index,
        "target":gt.values().flatten(),
    })
    for q in OUTPUT_QUANTILES:
        out[str(q)]=fc.quantile(q).values().flatten()
    out["prediction"]=out["0.5"]
    return out


In [ ]:
data=load_data(ROOT/"data")
df=data.full.sort_values("timestamp").copy()

series=TimeSeries.from_dataframe(
    df,time_col="timestamp",value_cols="target",freq="h"
).astype(np.float32)

future_covariates=TimeSeries.from_dataframe(
    df,time_col="timestamp",value_cols=FUTURE_PROXY_COLS,freq="h"
).astype(np.float32)

stat_train,_=series.split_after(TEST_CUTOFF)
future_train,_=future_covariates.split_after(TEST_CUTOFF)
test_start=TEST_CUTOFF+pd.Timedelta(hours=1)

print("Target:",series.start_time(),"->",series.end_time())
print("Future proxies:",future_covariates.start_time(),"->",future_covariates.end_time())
assert future_covariates.end_time()>=series.end_time()


In [ ]:
seasonal_naive=NaiveSeasonal(K=24)
seasonal_naive.fit(stat_train)

bt=seasonal_naive.historical_forecasts(
    series=series,
    start=test_start,
    forecast_horizon=24,
    stride=24,
    retrain=True,
    last_points_only=False,
    verbose=True,
)
forecast=concatenate(bt)
gt=series.slice_intersect(forecast)
fc=forecast.slice_intersect(gt)

result=pd.DataFrame({
    "item_id":1,
    "timestamp":gt.time_index,
    "target":gt.values().flatten(),
    "prediction":fc.values().flatten(),
})
result=clip_poa(result)

for q in OUTPUT_QUANTILES:
    result[str(q)]=np.nan

result.to_csv(OUTPUT_DIR/"seasonal_naive.csv",index=False)
print("Seasonal Naive")
print(f"MAE:  {mean_absolute_error(result.target,result.prediction):.6f}")
print(f"RMSE: {root_mean_squared_error(result.target,result.prediction):.6f}")
print("aQL:  N/A (deterministic baseline)")

power_result=convert_solar_result_to_power(result,data.test)
for q in OUTPUT_QUANTILES:
    if str(q) not in power_result.columns:
        power_result[str(q)]=np.nan
power_result.to_csv(OUTPUT_DIR/"seasonal_naive_power.csv",index=False)
print("Seasonal Naive [solar power]")
print(f"MAE:  {mean_absolute_error(power_result.target,power_result.prediction):.6f}")
print(f"RMSE: {root_mean_squared_error(power_result.target,power_result.prediction):.6f}")
print("aQL:  N/A (deterministic baseline)")


In [ ]:
for name in ["auto_arima","auto_theta","auto_ets"]:
    path=MODEL_DIR/f"{name}_solar_one_year.darts"

    if LOAD_EXISTING and path.exists():
        model=STAT_CLASSES[name].load(str(path))
    else:
        model=build_stat(name)
        model.fit(
            series=stat_train,
            future_covariates=future_train,
        )
        if SAVE_MODELS:
            model.save(str(path))

    bt=model.historical_forecasts(
        series=series,
        future_covariates=future_covariates,
        start=test_start,
        forecast_horizon=24,
        stride=24,
        num_samples=NUM_SAMPLES,
        retrain=False,
        last_points_only=False,
        verbose=True,
    )

    forecast=concatenate(bt)
    result=probabilistic_frame(series,forecast)

    # Match the original solar evaluation pipeline.
    result=clip_poa(result)

    result.to_csv(OUTPUT_DIR/f"{name}.csv",index=False)
    print_metrics(result,BASE_QUANTILES,label=name)

    power_result=convert_solar_result_to_power(result,data.test)
    power_result.to_csv(OUTPUT_DIR/f"{name}_power.csv",index=False)
    print_metrics(power_result,BASE_QUANTILES,label=f"{name} [solar power]")
